<!-- paired-role-banner -->
> 🟩 **정답 노트북 09 · 참고 코드 창**
>
> 왼쪽 실습 노트북과 같은 셀 순서입니다. 결과만 복사하기보다 각 shape, 자료형, 손실값과
> 설계 이유를 먼저 예측한 뒤 필요한 부분만 확인하세요.

# 09. Transformer from scratch: 텐서 shape로 이해하는 decoder

이 노트북은 작은 decoder-only Transformer를 **PyTorch 기본 연산부터 직접 조립**합니다. 완성 모델의 성능보다 각 연산의 입력·출력 shape와 causal 제약을 설명하는 것이 목표입니다.

학습 목표:

- token/position embedding이 합쳐지는 위치를 추적한다.
- Q, K, V와 multi-head 분할의 shape를 검증한다.
- scaled dot-product attention과 causal mask를 직접 구현한다.
- pre-LN residual block과 next-token loss를 연결한다.
- 저장소의 `TinyCausalTransformer`와 forward 계약을 비교한다.

CUDA → MPS → CPU를 자동 선택하며, 어느 장치에서도 수 초 안에 실행되도록 매우 작은 텐서만 사용합니다.

## 전체 데이터 흐름

표기 규칙은 `B=batch`, `T=time`, `D=model_dim`, `H=heads`, `Dh=D/H`, `V=vocab_size`입니다.

```text
token ids [B,T]
  → token embedding + position embedding [B,T,D]
  → Q/K/V [B,H,T,Dh]
  → attention score [B,H,T,T]
  → masked weighted sum [B,T,D]
  → residual + MLP [B,T,D]
  → vocabulary logits [B,T,V]
  → token t의 logit과 token t+1 label로 cross entropy
```

`assert`가 실패하면 다음 셀로 넘어가지 말고 실제 shape를 먼저 출력해 보세요.

<!-- api-explanations:start -->
<!-- api-explanations:keys=sys.path.insert,torch.manual_seed -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>sys.path.insert(index, path)</code></summary>

#### `sys.path.insert(index, path)`

- **역할:** Python 모듈 검색 경로의 지정 위치에 경로를 추가합니다.
- **입력·반환:** index와 경로 문자열을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 노트북에서 프로젝트 `src` 모듈을 import할 수 있게 합니다.
- **주의점:** 전역 import 상태를 바꾸므로 중복·우선순위와 실행 위치에 주의합니다.
- **공식 문서:** [sys.path.insert](https://docs.python.org/3/library/sys.html#sys.path)

</details>

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
import math
import sys
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "llm_engineering_lab").is_dir():
            return candidate
    raise RuntimeError("저장소 루트 또는 notebooks/에서 JupyterLab을 실행하세요.")


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "src"))
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(7)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
torch.set_default_device(DEVICE)
print(f"root={ROOT}\ntorch={torch.__version__} | {ACCELERATOR.summary()}")

## 1) 입력 token id와 padding mask

실제 토크나이저 대신 작은 정수 vocabulary를 사용해 모델 구조에 집중합니다. 두 번째 문장의 마지막 `0`은 padding입니다. padding은 key로 참조되지 않아야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.tensor,torch.bool -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>torch.bool</code></summary>

#### `torch.bool`

- **역할:** 불리언 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** mask와 조건 결과를 표현합니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.bool](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
PAD_ID, VOCAB_SIZE = 0, 16
BATCH_SIZE, SEQ_LEN, MODEL_DIM, NUM_HEADS = 2, 5, 12, 3
HEAD_DIM = MODEL_DIM // NUM_HEADS

input_ids = torch.tensor(
    [[1, 4, 5, 6, 2], [1, 7, 8, 2, PAD_ID]],
    dtype=torch.long,
    device=DEVICE,
)
attention_mask = input_ids.ne(PAD_ID)

assert input_ids.shape == (BATCH_SIZE, SEQ_LEN)
assert attention_mask.dtype == torch.bool
print("input_ids shape:", tuple(input_ids.shape))
print("attention_mask:\n", attention_mask.int())

## 2) token embedding + learned position embedding

token id는 의미를 학습하는 벡터로, 위치 id는 순서를 학습하는 벡터로 바뀝니다. 두 벡터의 차원이 같으므로 원소별로 더할 수 있습니다. 위치 정보가 없다면 self-attention 자체는 token 순서를 구분하지 못합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Embedding,torch.arange -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Embedding(num_embeddings, embedding_dim, ...)</code></summary>

#### `nn.Embedding(num_embeddings, embedding_dim, ...)`

- **역할:** 정수 ID를 학습 가능한 벡터로 조회합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
e_i=E[i],\qquad E\in\mathbb{R}^{V\times D}
$$

- **기호:** $i$는 정수 ID, $V$는 vocabulary 크기, $D$는 embedding 차원입니다.
- **수식 → 코드:** forward는 one-hot 행렬곱을 만들지 않고 weight 행 $E[i]$를 직접 조회합니다.
- **직관:** 범주 ID를 학습 가능한 연속 벡터 좌표로 바꿉니다.
- **shape 확인:** 입력 `[...] -> 출력 [..., D]`; 입력 dtype은 보통 `torch.int64`입니다.
- **공식 문서:** [nn.Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)

</details>

<details>
<summary><code>torch.arange(start=0, end, step=1, ...)</code></summary>

#### `torch.arange(start=0, end, step=1, ...)`

- **역할:** 일정 간격의 1차원 텐서를 만듭니다.
- **입력·반환:** 범위와 간격을 받고 1차원 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** position·index·time step을 벡터화해 만듭니다.
- **주의점:** 부동소수점 step은 반올림 때문에 원소 수가 예상과 다를 수 있습니다.
- **공식 문서:** [torch.arange](https://docs.pytorch.org/docs/stable/generated/torch.arange.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
token_embedding = nn.Embedding(VOCAB_SIZE, MODEL_DIM, padding_idx=PAD_ID)
position_embedding = nn.Embedding(SEQ_LEN, MODEL_DIM)
position_ids = torch.arange(SEQ_LEN, device=DEVICE).unsqueeze(0).expand(BATCH_SIZE, -1)

token_vectors = token_embedding(input_ids)
position_vectors = position_embedding(position_ids)
hidden = token_vectors + position_vectors

assert token_vectors.shape == (BATCH_SIZE, SEQ_LEN, MODEL_DIM)
assert position_vectors.shape == hidden.shape
print(
    "token / position / hidden:",
    token_vectors.shape,
    position_vectors.shape,
    hidden.shape,
)
print("padding embedding L2 norm:", token_embedding.weight[PAD_ID].norm().item())

## 3) 하나의 선형층에서 Q, K, V 만들기

실무 구현은 세 선형층 대신 `D → 3D` projection 하나를 사용한 뒤 마지막 축을 셋으로 나누는 경우가 많습니다. 그 다음 `D`를 `H × Dh`로 바꾸고 head 축을 앞으로 옮깁니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Linear,Tensor.view -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>tensor.view(*shape)</code></summary>

#### `tensor.view(*shape)`

- **역할:** 같은 저장공간을 보는 새로운 shape의 텐서를 만듭니다.
- **입력·반환:** 새 shape을 받고 view 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 연속 메모리의 축을 복사 없이 재해석합니다.
- **주의점:** stride가 호환되지 않으면 실패하므로 `reshape`나 `contiguous`가 필요합니다.

##### 관련 수식과 코드 연결

$$
\prod_{k=1}^{K} d_k=\prod_{j=1}^{J} d'_j
$$

- **기호:** $d_k$는 원래 각 축 크기, $d'_j$는 바꾼 뒤 각 축 크기입니다.
- **수식 → 코드:** `reshape`·`view`·`flatten`은 원소의 총개수를 보존하면서 축 해석만 바꿉니다.
- **직관:** 데이터 값을 계산하는 연산이 아니라 같은 원소를 다른 좌표계로 보는 연산입니다.
- **shape 확인:** 예: `[B, H, T, D] -> [B, T, H*D]`; 왼쪽과 오른쪽 원소 수가 같아야 합니다.
- **공식 문서:** [Tensor.view](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.view.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
qkv_projection = nn.Linear(MODEL_DIM, 3 * MODEL_DIM, bias=False)
packed_qkv = qkv_projection(hidden)
query, key, value = packed_qkv.chunk(3, dim=-1)


def split_heads(tensor: torch.Tensor) -> torch.Tensor:
    batch, time, model_dim = tensor.shape
    return tensor.view(batch, time, NUM_HEADS, HEAD_DIM).transpose(1, 2)


query, key, value = map(split_heads, (query, key, value))
expected = (BATCH_SIZE, NUM_HEADS, SEQ_LEN, HEAD_DIM)
assert query.shape == key.shape == value.shape == expected
print("packed QKV:", tuple(packed_qkv.shape))
print("split Q/K/V:", tuple(query.shape))

## 4) scaled dot-product attention score

각 query와 모든 key의 내적을 계산하면 `[B,H,T,T]`가 됩니다. `sqrt(Dh)`로 나누지 않으면 head dimension이 커질수록 내적 분산이 커져 softmax가 지나치게 뾰족해질 수 있습니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.transpose,torch.sqrt,Tensor.std -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.transpose(dim0, dim1)</code></summary>

#### `tensor.transpose(dim0, dim1)`

- **역할:** 두 축의 위치를 맞바꿉니다.
- **입력·반환:** 두 축 번호를 받고 축이 교환된 view를 반환합니다.
- **이 셀에서 쓰는 이유:** attention 행렬곱 등에서 축 순서를 맞춥니다.
- **주의점:** 반환 텐서는 비연속일 수 있어 이후 `view` 전에 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
Y_{i_{\pi(1)},\ldots,i_{\pi(K)}}=X_{i_1,\ldots,i_K}
$$

- **기호:** $\pi$는 축의 새 순서를 나타내는 permutation입니다.
- **수식 → 코드:** `transpose`는 두 축만 바꾸고 `permute`는 모든 축의 새 순서를 명시합니다.
- **직관:** 원소 값을 바꾸지 않고 각 index가 의미하는 축의 위치만 다시 배치합니다.
- **shape 확인:** 예: `[B,T,H,D] -> [B,H,T,D]`; 결과는 비연속 view일 수 있습니다.
- **공식 문서:** [Tensor.transpose](https://docs.pytorch.org/docs/stable/generated/torch.transpose.html)

</details>

<details>
<summary><code>torch.sqrt(input)</code></summary>

#### `torch.sqrt(input)`

- **역할:** 각 원소의 제곱근을 계산합니다.
- **입력·반환:** 0 이상 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 표준편차, 거리, diffusion schedule의 scale을 계산합니다.
- **주의점:** 음수 입력은 NaN을 만들며 0 근처 gradient가 매우 커질 수 있습니다.

##### 관련 수식과 코드 연결

$$
y_i=\sqrt{x_i},\qquad x_i\ge 0
$$

- **기호:** $x_i$는 음수가 아닌 입력입니다.
- **수식 → 코드:** API가 제곱 또는 분산 scale을 원래 단위의 길이로 되돌립니다.
- **직관:** 제곱된 크기를 사람이 해석하기 쉬운 원래 scale로 바꿉니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [torch.sqrt](https://docs.pytorch.org/docs/stable/generated/torch.sqrt.html)

</details>

<details>
<summary><code>tensor.std(dim=None, correction=1, keepdim=False)</code></summary>

#### `tensor.std(dim=None, correction=1, keepdim=False)`

- **역할:** 지정 축의 표준편차를 계산합니다.
- **입력·반환:** 텐서와 축·보정값을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 표현의 퍼짐, 데이터 scale, 학습 안정성을 진단합니다.
- **주의점:** 표본 수와 `correction`에 따라 작은 batch에서는 NaN이 될 수 있습니다.

##### 관련 수식과 코드 연결

$$
s=\sqrt{\frac{1}{N-\delta}\sum_{i=1}^{N}(x_i-\bar x)^2}
$$

- **기호:** $\delta$는 correction이며 NumPy 기본은 0, PyTorch `std` 기본은 1입니다.
- **수식 → 코드:** API의 `correction` 또는 `ddof`가 분모를 $N$과 $N-1$ 중 무엇으로 할지 정합니다.
- **직관:** 평균 주변에 값이 얼마나 퍼져 있는지를 원래 단위로 나타냅니다.
- **shape 확인:** 평균과 마찬가지로 지정한 축이 제거되거나 크기 1로 유지됩니다.
- **공식 문서:** [Tensor.std](https://docs.pytorch.org/docs/stable/generated/torch.std.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
raw_scores = query @ key.transpose(-2, -1)
scaled_scores = raw_scores / math.sqrt(HEAD_DIM)

assert scaled_scores.shape == (BATCH_SIZE, NUM_HEADS, SEQ_LEN, SEQ_LEN)
ratio = raw_scores.std().item() / max(scaled_scores.std().item(), 1e-12)
print("attention score shape:", tuple(scaled_scores.shape))
print(f"std ratio(raw/scaled)={ratio:.3f}, expected~sqrt(Dh)={math.sqrt(HEAD_DIM):.3f}")

## 5) causal mask와 padding mask

decoder LM의 위치 `t`는 `t+1` 이후 token을 볼 수 없어야 합니다. 아래 삼각 mask는 미래 key를 차단하고, padding mask는 실제 문장에 없는 key를 차단합니다. mask 적용 후 softmax 확률에서 미래 영역은 정확히 0이어야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.ones,Tensor.masked_fill,torch.softmax,torch.allclose,Tensor.sum -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.ones(*size, dtype=None, device=None)</code></summary>

#### `torch.ones(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 1인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 weight·mask·스케일을 구성합니다.
- **주의점:** 주변 연산과 dtype·device가 같은지 확인해야 합니다.
- **공식 문서:** [torch.ones](https://docs.pytorch.org/docs/stable/generated/torch.ones.html)

</details>

<details>
<summary><code>tensor.masked_fill(mask, value)</code></summary>

#### `tensor.masked_fill(mask, value)`

- **역할:** mask가 참인 위치를 지정 값으로 바꿉니다.
- **입력·반환:** 불리언 mask와 값을 받고 새 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** padding이나 미래 token의 attention score를 선택에서 제외합니다.
- **주의점:** mask는 입력에 broadcast 가능해야 하고 softmax 전 값은 충분히 작아야 합니다.

##### 관련 수식과 코드 연결

$$
s'_{ij}=\begin{cases}s_{ij},&M_{ij}=0\\-\infty,&M_{ij}=1\end{cases}\quad\Rightarrow\quad\operatorname{softmax}(s')_{ij}=0\;\text{if }M_{ij}=1
$$

- **기호:** $s$는 score, $M$은 제외할 위치가 참인 mask입니다.
- **수식 → 코드:** attention에서는 mask 위치를 매우 작은 값으로 채운 뒤 softmax를 적용합니다.
- **직관:** 확률을 나중에 지우는 대신 정규화 분모에도 들어가지 못하게 합니다.
- **shape 확인:** mask는 score에 broadcast 가능해야 하며 보통 `[B, H, T_q, T_k]`를 사용합니다.
- **공식 문서:** [Tensor.masked_fill](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.masked_fill.html)

</details>

<details>
<summary><code>torch.softmax(input, dim, dtype=None)</code></summary>

#### `torch.softmax(input, dim, dtype=None)`

- **역할:** 지정 축의 값을 합이 1인 확률형 가중치로 변환합니다.
- **입력·반환:** 텐서와 축을 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** attention score나 class logit을 정규화합니다.
- **주의점:** `dim`을 잘못 고르면 엉뚱한 축이 정규화되고 큰 logit은 포화될 수 있습니다.

##### 관련 수식과 코드 연결

$$
p_i=\frac{e^{z_i}}{\sum_j e^{z_j}},\qquad \log p_i=z_i-\log\sum_j e^{z_j}
$$

- **기호:** $z_i$는 logit, $p_i$는 선택한 축에서 합이 1인 확률입니다.
- **수식 → 코드:** 코드의 `dim`이 분모의 합을 계산할 $j$축을 결정합니다.
- **직관:** 각 logit을 다른 후보와 비교 가능한 상대 확률로 바꿉니다.
- **shape 확인:** 입력과 출력 shape은 같고 선택한 축의 softmax 값 합은 1입니다.
- **공식 문서:** [torch.softmax](https://docs.pytorch.org/docs/stable/generated/torch.softmax.html)

</details>

<details>
<summary><code>torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 텐서의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 두 텐서와 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 장치나 구현을 바꾼 뒤 수치 결과가 사실상 같은지 확인합니다.
- **주의점:** 정확한 동일성 검사가 아니며 작은 값에서는 `atol`을 직접 정해야 합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.allclose](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html)

</details>

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
causal_mask = torch.tril(torch.ones(SEQ_LEN, SEQ_LEN, dtype=torch.bool, device=DEVICE))
allowed = causal_mask.view(1, 1, SEQ_LEN, SEQ_LEN)
allowed = allowed & attention_mask.view(BATCH_SIZE, 1, 1, SEQ_LEN)

masked_scores = scaled_scores.masked_fill(
    ~allowed, torch.finfo(scaled_scores.dtype).min
)
attention_weights = F.softmax(masked_scores, dim=-1)

assert attention_weights.shape == scaled_scores.shape
assert torch.allclose(
    torch.triu(attention_weights, diagonal=1), torch.zeros_like(attention_weights)
)
assert torch.allclose(
    attention_weights.sum(dim=-1), torch.ones_like(attention_weights.sum(dim=-1))
)
print("causal mask:\n", causal_mask.int())
print(
    "head 0 attention weights (sample 0):\n", attention_weights[0, 0].round(decimals=3)
)

**관찰 포인트**

- score의 마지막 두 축에서 앞의 `T`는 query 위치, 뒤의 `T`는 key 위치입니다.
- padding query 자체를 0으로 만들지 않아도, padding 위치의 loss를 제외하면 학습 목표에는 포함되지 않습니다.
- 모든 key가 가려진 행을 만들면 softmax가 불안정해질 수 있으므로 실제 batch 계약을 명확히 해야 합니다.

**TODO 09-A:** `causal_mask`를 잠시 모두 `True`로 바꾼 뒤 미래 token을 수정했을 때 이전 위치 logit이 달라지는지 실험하세요.

## 6) multi-head attention 모듈로 묶기

이제 projection, head 분할, mask, weighted sum, output projection을 하나의 `nn.Module`로 묶습니다. 학습을 해부하기 위해 attention probability도 함께 반환합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Dropout -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Dropout(p=0.5)</code></summary>

#### `nn.Dropout(p=0.5)`

- **역할:** 학습 중 일부 activation을 무작위로 0으로 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\frac{m_i}{1-p}x_i,\qquad m_i\sim\operatorname{Bernoulli}(1-p)
$$

- **기호:** $p$는 제거 확률, $m_i$는 학습 중 표본화한 mask입니다.
- **수식 → 코드:** `model.train()`일 때 mask와 $1/(1-p)$ scale을 적용하고 `eval()`에서는 그대로 둡니다.
- **직관:** 일부 activation에 의존하지 못하게 하면서 기댓값은 유지합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class ScratchCausalSelfAttention(nn.Module):
    def __init__(self, model_dim: int, num_heads: int, dropout: float = 0.0) -> None:
        super().__init__()
        if model_dim % num_heads:
            raise ValueError("model_dim은 num_heads로 나누어져야 합니다.")
        self.num_heads = num_heads
        self.head_dim = model_dim // num_heads
        self.qkv = nn.Linear(model_dim, 3 * model_dim)
        self.out = nn.Linear(model_dim, model_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self, x: torch.Tensor, mask: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        batch, time, model_dim = x.shape
        query, key, value = self.qkv(x).chunk(3, dim=-1)

        def heads(tensor: torch.Tensor) -> torch.Tensor:
            return tensor.view(batch, time, self.num_heads, self.head_dim).transpose(
                1, 2
            )

        query, key, value = map(heads, (query, key, value))
        scores = query @ key.transpose(-2, -1) / math.sqrt(self.head_dim)
        causal = torch.tril(torch.ones(time, time, dtype=torch.bool, device=x.device))
        allowed_keys = causal[None, None] & mask[:, None, None, :].bool()
        scores = scores.masked_fill(~allowed_keys, torch.finfo(scores.dtype).min)
        weights = self.dropout(F.softmax(scores, dim=-1))
        context = weights @ value
        context = context.transpose(1, 2).contiguous().view(batch, time, model_dim)
        return self.out(context), weights

In [ ]:
attention = ScratchCausalSelfAttention(MODEL_DIM, NUM_HEADS)
attention_output, observed_weights = attention(hidden, attention_mask)

assert attention_output.shape == hidden.shape
assert observed_weights.shape == (BATCH_SIZE, NUM_HEADS, SEQ_LEN, SEQ_LEN)
future_max = torch.triu(observed_weights, diagonal=1).abs().max().detach().item()
assert future_max == 0.0
print("attention output:", tuple(attention_output.shape))
print("future attention max:", future_max)

## 7) pre-LN Transformer block

decoder block은 보통 다음 두 sub-layer로 구성됩니다.

1. `x = x + Attention(LayerNorm(x))`
2. `x = x + MLP(LayerNorm(x))`

residual 경로는 깊은 모델에서 gradient가 흐를 통로를 제공합니다. MLP는 각 token 위치에 독립적으로 적용됩니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.LayerNorm,nn.Sequential,nn.GELU -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.LayerNorm(normalized_shape, ...)</code></summary>

#### `nn.LayerNorm(normalized_shape, ...)`

- **역할:** 마지막 feature 축들을 표준화합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\gamma_i\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta_i
$$

- **기호:** $\mu,\sigma^2$는 한 sample의 지정 feature 축 평균·분산입니다.
- **수식 → 코드:** `normalized_shape`의 마지막 축들에서 통계를 내고 학습 가능한 scale과 bias를 적용합니다.
- **직관:** batch 크기와 무관하게 각 token·sample의 feature scale을 안정화합니다.
- **shape 확인:** 입력과 출력 shape은 같고 $\gamma,\beta$는 `normalized_shape`를 따릅니다.
- **공식 문서:** [nn.LayerNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)

</details>

<details>
<summary><code>nn.Sequential(*modules)</code></summary>

#### `nn.Sequential(*modules)`

- **역할:** 모듈을 순서대로 연결합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Sequential](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html)

</details>

<details>
<summary><code>nn.GELU(approximate=&#x27;none&#x27;)</code></summary>

#### `nn.GELU(approximate='none')`

- **역할:** 부드러운 GELU 활성함수를 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{GELU}(x)=x\Phi(x)\approx\frac{x}{2}\left[1+\tanh\!\left(\sqrt{\frac{2}{\pi}}(x+0.044715x^3)\right)\right]
$$

- **기호:** $\Phi(x)$는 표준정규분포의 누적분포함수입니다.
- **수식 → 코드:** `approximate` 설정에 따라 정확한 CDF 기반 값 또는 tanh 근사를 계산합니다.
- **직관:** 입력 크기에 비례해 부드럽게 gate하여 ReLU보다 경계가 매끄럽습니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.GELU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class ScratchTransformerBlock(nn.Module):
    def __init__(self, model_dim: int, num_heads: int, ff_dim: int) -> None:
        super().__init__()
        self.attn_norm = nn.LayerNorm(model_dim)
        self.attn = ScratchCausalSelfAttention(model_dim, num_heads)
        self.ff_norm = nn.LayerNorm(model_dim)
        self.ff = nn.Sequential(
            nn.Linear(model_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, model_dim),
        )

    def forward(
        self, x: torch.Tensor, mask: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        attended, weights = self.attn(self.attn_norm(x), mask)
        x = x + attended
        x = x + self.ff(self.ff_norm(x))
        return x, weights


block = ScratchTransformerBlock(MODEL_DIM, NUM_HEADS, ff_dim=24)
block_output, block_weights = block(hidden, attention_mask)
assert block_output.shape == hidden.shape
print("block output:", tuple(block_output.shape))

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.backward,torch.linalg.norm,Tensor.item -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>torch.linalg.vector_norm(x, ord=2, dim=None, keepdim=False)</code></summary>

#### `torch.linalg.vector_norm(x, ord=2, dim=None, keepdim=False)`

- **역할:** 벡터의 크기인 norm을 계산합니다.
- **입력·반환:** 텐서·차수·축을 받고 norm 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 거리, gradient 크기, 정규화 분모를 계산합니다.
- **주의점:** `dim`과 norm 차수에 따라 의미와 반환 shape이 달라집니다.

##### 관련 수식과 코드 연결

$$
\lVert x\rVert_p=\left(\sum_i\lvert x_i\rvert^p\right)^{1/p},\qquad \lVert x\rVert_2=\sqrt{\sum_i x_i^2}
$$

- **기호:** $p$는 norm 차수이며 $p=2$가 유클리드 길이입니다.
- **수식 → 코드:** `ord`와 `dim`·`axis`가 어떤 원소 묶음의 크기를 계산할지 정합니다.
- **직관:** 여러 성분을 하나의 거리·크기 값으로 요약합니다.
- **shape 확인:** 지정한 축은 결과에서 제거되며 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [torch.linalg.norm](https://docs.pytorch.org/docs/stable/generated/torch.linalg.vector_norm.html)

</details>

<details>
<summary><code>tensor.item()</code></summary>

#### `tensor.item()`

- **역할:** 원소 하나짜리 텐서에서 Python 숫자를 꺼냅니다.
- **입력·반환:** 단일 원소 텐서를 받고 Python 숫자를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 metric을 출력·기록 가능한 값으로 바꿉니다.
- **주의점:** 여러 원소 텐서에는 쓸 수 없고 GPU에서는 동기화를 일으킬 수 있습니다.
- **공식 문서:** [Tensor.item](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.item.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# gradient가 embedding과 QKV projection까지 실제로 흐르는지 확인합니다.
diagnostic_loss = block_output.square().mean()
diagnostic_loss.backward()
grad_norm = block.attn.qkv.weight.grad.norm().item()

assert grad_norm > 0
print(f"diagnostic_loss={diagnostic_loss.item():.4f}, qkv_grad_norm={grad_norm:.4f}")

## 8) vocabulary projection과 next-token loss

각 위치의 hidden state를 vocabulary 크기의 logit으로 바꿉니다. teacher forcing에서는 입력과 label을 한 칸 이동합니다.

- `logits[:, 0]`은 `labels[:, 1]`을 예측
- `logits[:, T-2]`는 `labels[:, T-1]`을 예측
- padding label은 loss에서 제외

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.ModuleList,F.cross_entropy -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.ModuleList(modules=None)</code></summary>

#### `nn.ModuleList(modules=None)`

- **역할:** 하위 모듈 목록을 등록 가능한 형태로 보관합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.ModuleList](https://docs.pytorch.org/docs/stable/generated/torch.nn.ModuleList.html)

</details>

<details>
<summary><code>F.cross_entropy(input, target, ...)</code></summary>

#### `F.cross_entropy(input, target, ...)`

- **역할:** logit에 log-softmax와 음의 로그우도를 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{CE}=-\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{c=1}^{C}e^{z_{n,c}}}
$$

- **기호:** $z_{n,c}$는 class logit, $y_n$은 정답 class ID, $C$는 class 수입니다.
- **수식 → 코드:** 함수가 내부에서 `log_softmax`와 정답 class의 음의 로그우도를 결합합니다.
- **직관:** 정답 class 확률이 낮을수록 큰 벌점을 줍니다.
- **shape 확인:** 일반적으로 logits `[N,C,...]`, class-index target `[N,...]`을 사용합니다.
- **공식 문서:** [F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class ScratchTinyLM(nn.Module):
    def __init__(self, vocab_size: int, pad_id: int, max_length: int) -> None:
        super().__init__()
        self.pad_id = pad_id
        self.token_embedding = nn.Embedding(vocab_size, MODEL_DIM, padding_idx=pad_id)
        self.position_embedding = nn.Embedding(max_length, MODEL_DIM)
        self.blocks = nn.ModuleList(
            [ScratchTransformerBlock(MODEL_DIM, NUM_HEADS, ff_dim=24)]
        )
        self.norm = nn.LayerNorm(MODEL_DIM)
        self.lm_head = nn.Linear(MODEL_DIM, vocab_size, bias=False)

    def forward(
        self,
        token_ids: torch.Tensor,
        mask: torch.Tensor,
        labels: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        batch, time = token_ids.shape
        positions = torch.arange(time, device=token_ids.device)[None, :].expand(
            batch, -1
        )
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        for block in self.blocks:
            x, _ = block(x, mask)
        logits = self.lm_head(self.norm(x))
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=self.pad_id,
            )
        return logits, loss

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.optim.AdamW,Module.parameters,torch.isfinite,Optimizer.zero_grad,Optimizer.step -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-2, ...)</code></summary>

#### `torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-2, ...)`

- **역할:** weight decay를 gradient update와 분리한 AdamW optimizer를 만듭니다.
- **입력·반환:** 파라미터와 hyperparameter를 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** Transformer 계열에서 흔한 최적화 규칙을 재현합니다.
- **주의점:** bias·정규화 파라미터를 decay에서 제외하는 설정도 자주 사용합니다.

##### 관련 수식과 코드 연결

$$
\theta_t=(1-\eta\lambda)\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}
$$

- **기호:** $\lambda$는 weight decay이고 나머지 기호는 Adam과 같습니다.
- **수식 → 코드:** 코드의 `weight_decay`가 gradient 기반 Adam update와 분리된 축소 항으로 적용됩니다.
- **직관:** 손실 gradient와 무관하게 큰 weight를 조금씩 줄이는 것이 Adam과의 핵심 차이입니다.
- **shape 확인:** 파라미터와 모멘트 tensor의 shape은 변하지 않습니다.
- **공식 문서:** [torch.optim.AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>torch.isfinite(input)</code></summary>

#### `torch.isfinite(input)`

- **역할:** 텐서의 각 원소가 유한수인지 검사합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 불리언 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 gradient의 NaN·무한대 발생을 빠르게 찾습니다.
- **주의점:** 검사 결과는 미분 대상이 아니며 원인 자체를 고치지는 않습니다.
- **공식 문서:** [torch.isfinite](https://docs.pytorch.org/docs/stable/generated/torch.isfinite.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
scratch_lm = ScratchTinyLM(VOCAB_SIZE, PAD_ID, SEQ_LEN).to(DEVICE)
optimizer = torch.optim.AdamW(scratch_lm.parameters(), lr=1e-3)
logits, next_token_loss = scratch_lm(input_ids, attention_mask, labels=input_ids)

assert logits.shape == (BATCH_SIZE, SEQ_LEN, VOCAB_SIZE)
assert next_token_loss is not None and torch.isfinite(next_token_loss)
optimizer.zero_grad(set_to_none=True)
next_token_loss.backward()
optimizer.step()
print(
    "logits:", tuple(logits.shape), "next-token loss:", round(next_token_loss.item(), 4)
)

## 9) 저장소 구현과 forward 계약 비교

직접 만든 모델은 개념 확인용입니다. 저장소 구현은 입력 검증, dropout, weight tying, KV cache, generation까지 포함합니다. 내부 수치가 같을 필요는 없지만 `[B,T] → [B,T,V]`와 next-token loss 계약은 같아야 합니다.

In [ ]:
from llm_engineering_lab.transformer import TinyCausalTransformer, TinyTransformerConfig

production_shaped = TinyCausalTransformer(
    TinyTransformerConfig(
        vocab_size=VOCAB_SIZE,
        pad_id=PAD_ID,
        max_seq_length=SEQ_LEN,
        model_dim=MODEL_DIM,
        num_heads=NUM_HEADS,
        num_layers=1,
        feed_forward_dim=24,
        dropout=0.0,
    )
).to(DEVICE)
result = production_shaped(input_ids, attention_mask=attention_mask, labels=input_ids)

assert result.logits.shape == logits.shape
assert result.loss is not None and torch.isfinite(result.loss)
print(
    "repo logits:",
    tuple(result.logits.shape),
    "repo loss:",
    round(result.loss.item(), 4),
)
print("parameter count:", sum(p.numel() for p in production_shaped.parameters()))

## 마무리 도전 과제

- **TODO 09-B (필수):** `ScratchCausalSelfAttention`에 attention dropout을 추가하고 `train()`/`eval()` 결과 차이를 고정 seed로 확인하세요.
- **TODO 09-C (shape):** `B=3, T=7, D=24, H=6`으로 바꾸고 모든 assert가 통과하게 만드세요.
- **TODO 09-D (누수 테스트):** 미래 token만 바꾼 두 입력에서 이전 위치 logits가 동일한지 `torch.testing.assert_close`로 검증하세요.
- **TODO 09-E (심화):** learned position embedding을 sinusoidal 또는 RoPE로 교체하고 장단점을 기록하세요.
- **TODO 09-F (현업):** attention probability 전체를 저장할 때 필요한 메모리를 `B×H×T×T×dtype_bytes`로 계산하세요.

완료 기준: 각 단계의 shape와 causal mask가 필요한 이유를 그림 없이도 설명할 수 있습니다.